In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 1. 환경 설정 (Drive 마운트 + 패키지 설치 + 캐시를 Drive로)
# ═══════════════════════════════════════════════════════════════════
import os, subprocess, sys, shutil

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR  = "/content/drive/MyDrive/dacon"
WORK_DIR   = "/content/dacon"
LOCAL_DATA = os.path.join(WORK_DIR, "data")
CACHE_DIR  = os.path.join(DRIVE_DIR, ".cache")
LOCAL_CKPT = "/content/dacon/local_ckpt"

for d in [
    DRIVE_DIR,
    os.path.join(DRIVE_DIR, "checkpoints"),
    os.path.join(DRIVE_DIR, "submissions"),
    CACHE_DIR, WORK_DIR, LOCAL_CKPT,
]:
    os.makedirs(d, exist_ok=True)

os.environ["HF_HOME"] = os.path.join(CACHE_DIR, "huggingface")
os.environ["TRANSFORMERS_CACHE"] = os.path.join(CACHE_DIR, "huggingface")
os.environ["TORCH_HOME"] = os.path.join(CACHE_DIR, "torch")
os.environ["XDG_CACHE_HOME"] = CACHE_DIR
os.environ["PIP_CACHE_DIR"] = os.path.join(CACHE_DIR, "pip")
os.environ["TMPDIR"] = os.path.join(CACHE_DIR, "tmp")
os.makedirs(os.environ["TMPDIR"], exist_ok=True)
os.environ["DACON_LOCAL_CKPT"] = LOCAL_CKPT

import torch
if torch.cuda.is_available():
    gpu  = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu} ({vram:.1f} GB)")
else:
    print("GPU 없음! 런타임 > 런타임 유형 변경 > GPU 선택")

free = shutil.disk_usage('/content').free / 1e9
print(f"로컬 디스크 여유: {free:.1f} GB")
print(f"Resume ckpt 경로: {LOCAL_CKPT} (로컬, 세션 종료 시 삭제)")
print(f"Best model 경로: {DRIVE_DIR}/checkpoints/ (Drive, 영구)")

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "timm>=1.0.20", "albumentations>=1.4.0",
    "opencv-python-headless", "scikit-learn", "pandas", "tqdm", "xgboost",
])
print("패키지 설치 완료 (timm, albumentations, sklearn, xgboost)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 2. 소스 파일 복사 + 체크포인트 설정 (로컬 저장 -> Drive 동기화)
# ═══════════════════════════════════════════════════════════════════
import os, shutil

DRIVE_DIR = "/content/drive/MyDrive/dacon"
WORK_DIR  = "/content/dacon"
DRIVE_CKPT = os.path.join(DRIVE_DIR, "checkpoints")
LOCAL_CKPT_BEST = os.path.join(WORK_DIR, "checkpoints")
DRIVE_SUB  = os.path.join(DRIVE_DIR, "submissions")

# --- 소스 파일 복사 ---
SRC_FILES = ["models.py", "datasets.py", "train.py", "train_v2.py", "inference.py", "inference_v2.py"]
for f in SRC_FILES:
    src = os.path.join(DRIVE_DIR, f)
    dst = os.path.join(WORK_DIR, f)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"  [OK] {f}")
    else:
        print(f"  [MISSING] {f} - Drive에 업로드 필요: {DRIVE_DIR}/")

# --- 체크포인트: 로컬 디렉토리 (빠른 저장) ---
os.makedirs(LOCAL_CKPT_BEST, exist_ok=True)

# --- submissions: Drive 직접 ---
os.makedirs(DRIVE_SUB, exist_ok=True)
sub_link = os.path.join(WORK_DIR, "submissions")
if os.path.islink(sub_link):
    os.unlink(sub_link)
elif os.path.isdir(sub_link):
    shutil.rmtree(sub_link)
os.symlink(DRIVE_SUB, sub_link)
print(f"  submissions/ -> Drive (영구)")

# --- Drive에 기존 best 가중치 있으면 로컬로 복사 (resume용) ---
os.makedirs(DRIVE_CKPT, exist_ok=True)
copied = 0
for f in sorted(os.listdir(DRIVE_CKPT)):
    if not f.endswith('.pth'):
        continue
    src = os.path.join(DRIVE_CKPT, f)
    dst = os.path.join(LOCAL_CKPT_BEST, f)
    if not os.path.exists(dst):
        mb = os.path.getsize(src) / 1e6
        print(f"  <- Drive->로컬: {f} ({mb:.0f} MB)")
        shutil.copy2(src, dst)
        copied += 1
if copied == 0:
    print("  Drive 체크포인트 없음 또는 이미 로컬에 존재")

# --- 동기화 헬퍼 함수 ---
def sync_best_to_drive(backbone=None):
    """로컬 best 가중치 -> Drive 복사 (ckpt 제외, best만)"""
    count = 0
    for f in sorted(os.listdir(LOCAL_CKPT_BEST)):
        if not f.endswith('.pth'):
            continue
        if '_ckpt' in f:
            continue
        if backbone and not f.startswith(backbone):
            continue
        src = os.path.join(LOCAL_CKPT_BEST, f)
        dst = os.path.join(DRIVE_CKPT, f)
        src_size = os.path.getsize(src)
        if os.path.exists(dst) and os.path.getsize(dst) == src_size:
            continue
        mb = src_size / 1e6
        print(f"  -> Drive: {f} ({mb:.0f} MB)")
        shutil.copy2(src, dst)
        count += 1
    if count == 0:
        print("  동기화할 새 가중치 없음")
    else:
        print(f"  {count}개 가중치 Drive 동기화 완료")

# --- 현황 ---
print(f"\n체크포인트 저장: 로컬 ({LOCAL_CKPT_BEST})")
for bb in ["dinov2_large", "eva_giant"]:
    ckpts = [f for f in os.listdir(LOCAL_CKPT_BEST)
             if f.startswith(bb) and f.endswith('.pth') and '_ckpt' not in f]
    print(f"  {bb}: {len(ckpts)}개 best 가중치")
    for c in sorted(ckpts):
        mb = os.path.getsize(os.path.join(LOCAL_CKPT_BEST, c)) / 1e6
        print(f"    {c} ({mb:.0f} MB)")
print(f"\nDrive 체크포인트: {DRIVE_CKPT}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 3. 코랩 경쟁 모드 패치 (강한 증강 + 6-fold TTA)
# ═══════════════════════════════════════════════════════════════════
from pathlib import Path

path = Path("/content/dacon/datasets.py")
text = path.read_text(encoding="utf-8")

old_train = '''def get_train_transforms(img_size=384):
    \"\"\"경쟁급 최대 증강\"\"\"
    return A.Compose([
        A.RandomResizedCrop(
            (img_size, img_size), scale=(0.8, 1.0), ratio=(0.9, 1.1), p=1.0),
        A.HorizontalFlip(p=0.5),
        A.Affine(
            translate_percent=(-0.05, 0.05), scale=(0.93, 1.07),
            rotate=(-10, 10), p=0.5),
        A.Perspective(scale=(0.02, 0.06), p=0.3),
        A.OneOf([
            A.RandomBrightnessContrast(
                brightness_limit=0.2, contrast_limit=0.2, p=1.0),
            A.HueSaturationValue(
                hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=15, p=1.0),
            A.ColorJitter(
                brightness=0.2, contrast=0.2, saturation=0.15, hue=0.05, p=1.0),
        ], p=0.8),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 7), p=1.0),
            A.MotionBlur(blur_limit=(3, 7), p=1.0),
        ], p=0.2),
        A.OneOf([
            A.GaussNoise(std_range=(0.02, 0.10), p=1.0),
            A.ISONoise(p=1.0),
        ], p=0.2),
        A.RandomShadow(p=0.15),
        A.RandomFog(fog_coef_range=(0.1, 0.35), p=0.1),
        A.CoarseDropout(
            num_holes_range=(1, 6),
            hole_height_range=(16, 40), hole_width_range=(16, 40), p=0.25),
        A.ToGray(p=0.05),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])'''

new_train = '''def get_train_transforms(img_size=384):
    \"\"\"코랩 경쟁 모드용 강화 증강\"\"\"
    return A.Compose([
        A.RandomResizedCrop(
            (img_size, img_size), scale=(0.72, 1.0), ratio=(0.88, 1.12), p=1.0),
        A.HorizontalFlip(p=0.5),
        A.Affine(
            translate_percent=(-0.08, 0.08), scale=(0.90, 1.10),
            rotate=(-12, 12), p=0.6),
        A.Perspective(scale=(0.03, 0.08), p=0.35),
        A.OneOf([
            A.RandomBrightnessContrast(
                brightness_limit=0.25, contrast_limit=0.25, p=1.0),
            A.HueSaturationValue(
                hue_shift_limit=12, sat_shift_limit=18, val_shift_limit=18, p=1.0),
            A.ColorJitter(
                brightness=0.25, contrast=0.25, saturation=0.20, hue=0.06, p=1.0),
            A.CLAHE(clip_limit=(1, 4), p=1.0),
        ], p=0.9),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 7), p=1.0),
            A.MotionBlur(blur_limit=(3, 7), p=1.0),
            A.Sharpen(alpha=(0.1, 0.3), lightness=(0.8, 1.1), p=1.0),
        ], p=0.25),
        A.OneOf([
            A.GaussNoise(std_range=(0.02, 0.12), p=1.0),
            A.ISONoise(p=1.0),
            A.ImageCompression(quality_range=(45, 90), p=1.0),
        ], p=0.25),
        A.RandomShadow(p=0.20),
        A.RandomFog(fog_coef_range=(0.1, 0.35), p=0.12),
        A.CoarseDropout(
            num_holes_range=(1, 8),
            hole_height_range=(12, 56), hole_width_range=(12, 56), p=0.35),
        A.ToGray(p=0.08),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])'''

old_tta = '''def get_tta_transforms(img_size=384):
    norm = dict(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    crop = int(img_size * 0.9)
    return [
        # 0: base (center crop)
        A.Compose([A.CenterCrop(crop, crop),
                    A.Resize(img_size, img_size), A.Normalize(**norm), ToTensorV2()]),
        # 1: hflip
        A.Compose([A.CenterCrop(crop, crop),
                    A.Resize(img_size, img_size), A.HorizontalFlip(p=1.0),
                    A.Normalize(**norm), ToTensorV2()]),
        # 2: brightness
        A.Compose([A.CenterCrop(crop, crop),
                    A.Resize(img_size, img_size),
                    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=1.0),
                    A.Normalize(**norm), ToTensorV2()]),
        # 3: tighter crop (90% of 90% = 81%)
        A.Compose([A.CenterCrop(int(img_size * 0.9), int(img_size * 0.9)),
                    A.Resize(img_size, img_size), A.Normalize(**norm), ToTensorV2()]),
    ]'''

new_tta = '''def get_tta_transforms(img_size=384):
    norm = dict(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    crop = int(img_size * 0.9)
    return [
        A.Compose([A.CenterCrop(crop, crop),
                    A.Resize(img_size, img_size), A.Normalize(**norm), ToTensorV2()]),
        A.Compose([A.CenterCrop(crop, crop),
                    A.Resize(img_size, img_size), A.HorizontalFlip(p=1.0),
                    A.Normalize(**norm), ToTensorV2()]),
        A.Compose([A.CenterCrop(crop, crop),
                    A.Resize(img_size, img_size),
                    A.RandomBrightnessContrast(brightness_limit=0.10, contrast_limit=0.10, p=1.0),
                    A.Normalize(**norm), ToTensorV2()]),
        A.Compose([A.CenterCrop(crop, crop),
                    A.Resize(img_size, img_size),
                    A.CLAHE(clip_limit=(1, 3), p=1.0),
                    A.Normalize(**norm), ToTensorV2()]),
        A.Compose([A.CenterCrop(crop, crop),
                    A.Resize(img_size, img_size),
                    A.Affine(scale=(0.97, 1.03), translate_percent=(-0.02, 0.02), rotate=(-3, 3), p=1.0),
                    A.Normalize(**norm), ToTensorV2()]),
        A.Compose([A.CenterCrop(int(img_size * 0.9), int(img_size * 0.9)),
                    A.Resize(img_size, img_size), A.Normalize(**norm), ToTensorV2()]),
    ]'''

if old_train in text:
    text = text.replace(old_train, new_train)
else:
    print('[WARN] train transform block not matched')

if old_tta in text:
    text = text.replace(old_tta, new_tta)
else:
    print('[WARN] tta transform block not matched')

path.write_text(text, encoding='utf-8')
print('코랩 런타임용 datasets.py 패치 완료')
print('  - train augmentation 강화 (crop 0.72~1.0, CLAHE, Sharpen, Compression)')
print('  - TTA 6개 (전부 CenterCrop 90% 적용)')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 4. 데이터 압축 해제 (Drive data.zip → 로컬 SSD)
# ═══════════════════════════════════════════════════════════════════
import os, shutil, zipfile, time

DRIVE_DIR  = "/content/drive/MyDrive/dacon"
WORK_DIR   = "/content/dacon"
LOCAL_DATA = os.path.join(WORK_DIR, "data")
ZIP_PATH   = os.path.join(DRIVE_DIR, "data.zip")

if os.path.islink(LOCAL_DATA):
    os.unlink(LOCAL_DATA)
elif os.path.isdir(LOCAL_DATA):
    shutil.rmtree(LOCAL_DATA)
os.makedirs(LOCAL_DATA, exist_ok=True)

if not os.path.exists(ZIP_PATH):
    print(f"[ERROR] {ZIP_PATH} 를 찾을 수 없습니다.")
    print(f"  Drive에 data.zip을 업로드하세요: {DRIVE_DIR}/data.zip")
else:
    zip_mb = os.path.getsize(ZIP_PATH) / 1e6
    print(f"data.zip 발견: {zip_mb:.0f} MB")
    print(f"해제 중... (로컬 SSD → 빠름)")
    t0 = time.time()
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(LOCAL_DATA)
    elapsed = time.time() - t0
    print(f"  해제 완료: {elapsed:.1f}초")

for f in ["open/train.csv", "open/dev.csv", "open/sample_submission.csv"]:
    p = os.path.join(LOCAL_DATA, f)
    ok = "[OK]" if os.path.exists(p) else "[FAIL]"
    print(f"  {ok} {f}")

for split in ["train", "dev", "test"]:
    d = os.path.join(LOCAL_DATA, "open", split)
    if os.path.isdir(d):
        n = len([x for x in os.listdir(d) if os.path.isdir(os.path.join(d, x))])
        print(f"  {split}/: {n}개")

ss = os.path.join(LOCAL_DATA, "shapestacks", "shapestacks", "recordings")
if os.path.exists(ss):
    h6 = [d for d in os.listdir(ss) if "h=6" in d]
    print(f"  ShapeStacks h=6: {len(h6)} scenarios")
else:
    print("  [SKIP] ShapeStacks 없음 (Dacon only)")

free = shutil.disk_usage('/content').free / 1e9
print(f"\n로컬 디스크 여유: {free:.1f} GB")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 5. 환경 검증 (DINOv2 + EVA-Giant 양쪽 체크포인트 확인)
# ═══════════════════════════════════════════════════════════════════
import os, shutil

WORK_DIR  = "/content/dacon"
DRIVE_DIR = "/content/drive/MyDrive/dacon"

print("=" * 60)
print("  v10 환경 검증 (DINOv2 + EVA-Giant)")
print("=" * 60)

for f in ["models.py", "datasets.py", "train.py", "train_v2.py", "inference.py", "inference_v2.py"]:
    ok = os.path.exists(os.path.join(WORK_DIR, f))
    print(f"  {'[OK]' if ok else '[FAIL]'} {f}")

tc = os.path.join(WORK_DIR, "data", "open", "train.csv")
print(f"  {'[OK]' if os.path.exists(tc) else '[FAIL]'} train.csv")
dc = os.path.join(WORK_DIR, "data", "open", "dev.csv")
print(f"  {'[OK]' if os.path.exists(dc) else '[FAIL]'} dev.csv")

ckpt_dir = os.path.join(WORK_DIR, "checkpoints")
is_local = os.path.isdir(ckpt_dir) and not os.path.islink(ckpt_dir)
print(f"  {'[OK]' if is_local else '[WARN]'} checkpoints/ (로컬 저장)")

sub_link = os.path.join(WORK_DIR, "submissions")
is_drive = os.path.islink(sub_link) and "drive" in os.readlink(sub_link)
print(f"  {'[OK]' if is_drive else '[FAIL]'} submissions/ -> Drive")

# --- 백본별 체크포인트 현황 ---
print()
for bb_name in ["dinov2_large", "eva_giant"]:
    ckpts = [f for f in os.listdir(ckpt_dir) if f.startswith(bb_name) and f.endswith('.pth')] if os.path.isdir(ckpt_dir) else []
    best_ckpts = sorted([c for c in ckpts if '_ckpt' not in c])
    resume_ckpts = sorted([c for c in ckpts if '_ckpt' in c])
    fold_status = []
    for fold in range(5):
        exists = f"{bb_name}_fold{fold}.pth" in best_ckpts
        fold_status.append("✓" if exists else "✗")
    print(f"  {bb_name}: Folds [{' '.join(fold_status)}]  ({len(best_ckpts)} best, {len(resume_ckpts)} resume)")
    for c in best_ckpts:
        mb = os.path.getsize(os.path.join(ckpt_dir, c)) / 1e6
        print(f"    {c} ({mb:.0f} MB)")

# Drive 현황
drive_ckpt = os.path.join(DRIVE_DIR, "checkpoints")
drive_ckpts = sorted([f for f in os.listdir(drive_ckpt) if f.endswith('.pth')]) if os.path.isdir(drive_ckpt) else []
if drive_ckpts:
    print(f"\n  Drive 백업: {len(drive_ckpts)}개")

free = shutil.disk_usage('/content').free / 1e9
print(f"\n  디스크 여유: {free:.1f} GB")
print("=" * 60)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 6. DINOv2 학습 설정 (Stage 1: 구조물 인식 전문가)
# ═══════════════════════════════════════════════════════════════════
# DINOv2-Large: self-supervised pretrained → spatial structure 이해 우수
# cross_attn head: front/top 토큰을 Transformer로 교차 융합 → 구조 인식 극대화
# (기존 로컬 학습 설정과 동일하게 맞춤)
# ═══════════════════════════════════════════════════════════════════
import os
os.chdir("/content/dacon")

DINO_CONFIG = {
    "backbone": "dinov2_large",
    "epochs": 20,
    "patience": 10,
    "head_lr": 1e-4,          # cross_attn head는 파라미터 많음 → 높은 LR
    "bb_lr": 1e-5,
    "weight_decay": 0.05,
    "drop_rate": 0.3,
    "layer_decay": 0.75,      # DINOv2 최적값 (로컬 검증 완료)
    "warmup_epochs": 2,
    "merge_dev": True,
    "loss": "ce",
    "no_mixup": True,
    "simple_aug": True,
    "head_type": "cross_attn", # CrossViewFusion: front/top 토큰 교차 attention
    "scheduler": "cosine",
    "batch_size": 16,
}

EVA_CONFIG = None  # Cell 10에서 정의

# 백본별 HEAD_TYPE 매핑 (분석/추론 셀에서 공통 사용)
HEAD_TYPE_MAP = {
    "dinov2_large": "cross_attn",
    "eva_giant": "simple",
}

# ═══════════════════════════════════════════════════════════════════
# 공통 학습 헬퍼 함수 (DINOv2 / EVA 양쪽에서 사용)
# ═══════════════════════════════════════════════════════════════════
def run_finetune_fold(cfg, fold, sync_fn=None):
    """단일 fold finetune 학습을 subprocess로 실행"""
    import os, sys, shlex, subprocess, shutil
    os.chdir("/content/dacon")
    backbone = cfg["backbone"]

    cmd = [
        sys.executable, "-u", "train.py",
        "--backbone", backbone,
        "--stage", "finetune",
        "--finetune_epochs", str(cfg["epochs"]),
        "--patience", str(cfg["patience"]),
        "--fold", str(fold),
        "--loss", cfg.get("loss", "ce"),
        "--scheduler", cfg.get("scheduler", "cosine"),
        "--head_type", cfg.get("head_type", "simple"),
        "--head_lr", str(cfg["head_lr"]),
        "--bb_lr", str(cfg["bb_lr"]),
        "--weight_decay", str(cfg["weight_decay"]),
        "--drop_rate", str(cfg["drop_rate"]),
        "--layer_decay", str(cfg["layer_decay"]),
        "--warmup_epochs", str(cfg.get("warmup_epochs", 2)),
        "--grad_checkpointing", "--resume", "--init_from_best",
        "--num_workers", "0",
    ]
    if cfg.get("merge_dev"):   cmd += ["--merge_dev"]
    if cfg.get("no_mixup"):    cmd += ["--no_mixup"]
    if cfg.get("simple_aug"):  cmd += ["--simple_aug"]
    cmd += ["--batch_size_override", str(cfg.get("batch_size", 16))]

    cmd_str = shlex.join(cmd)
    print(f"CMD: {cmd_str}\n" + "=" * 60)

    env = os.environ.copy()
    env["TQDM_FORCE_TTY"] = "1"
    process = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=env
    )
    while True:
        char = process.stdout.read(1)
        if not char and process.poll() is not None:
            break
        sys.stdout.write(char)
        sys.stdout.flush()
    process.wait()

    if process.returncode != 0:
        print(f"\n[ERROR] {backbone} Fold {fold} 실패 (exit code {process.returncode})")
    else:
        free = shutil.disk_usage('/content').free / 1e9
        print(f"\n디스크 여유: {free:.1f} GB")
        if sync_fn:
            print("Drive 동기화...")
            sync_fn(backbone)

# --- 설정 출력 ---
print(f"{'='*60}")
print(f"  DINOv2-Large 학습 설정 (구조물 인식 전문가)")
print(f"{'='*60}")
for k, v in DINO_CONFIG.items():
    print(f"  {k:>16}: {v}")
print(f"\n  [핵심] cross_attn head: front/top 양쪽 뷰의 패치 토큰을")
print(f"         Transformer로 교차 융합 → 3D 구조 인식 극대화")
print(f"  [특성] DINOv2는 self-supervised spatial feature 학습")
print(f"  → 건물 구조 인식에 강하나 안정성 세부 분류는 약함")

for fold in range(5):
    best = os.path.join("checkpoints", f"dinov2_large_fold{fold}.pth")
    status = "✓ 완료" if os.path.exists(best) else "✗ 미완료"
    print(f"  Fold {fold}: {status}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 7. DINOv2 Stage 1: Pretrain (선택) — 경쟁 모드에서는 기본 건너뜀
# ═══════════════════════════════════════════════════════════════════
import os, sys, shlex, subprocess
os.chdir("/content/dacon")

DINO_PRETRAIN_EPOCHS = 0  # 0이면 건너뜀

if DINO_PRETRAIN_EPOCHS <= 0:
    print("경쟁 모드 기본값: DINOv2 pretrain 생략")
else:
    cmd = [
        sys.executable, "-u", "train.py",
        "--backbone", "dinov2_large",
        "--stage", "pretrain",
        "--pretrain_epochs", str(DINO_PRETRAIN_EPOCHS),
        "--batch_size_override", "16",
        "--grad_checkpointing", "--resume",
        "--num_workers", "0",
    ]
    cmd_str = shlex.join(cmd)
    print(f"CMD: {cmd_str}\n" + "=" * 50)
    env = os.environ.copy()
    env["TQDM_FORCE_TTY"] = "1"
    process = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=env)
    while True:
        char = process.stdout.read(1)
        if not char and process.poll() is not None: break
        sys.stdout.write(char); sys.stdout.flush()
    process.wait()
    if process.returncode != 0:
        print(f"\n[ERROR] DINOv2 Pretrain 실패 (exit code {process.returncode})")

In [ ]:
# 8-0. DINOv2 Finetune Fold 0
run_finetune_fold(DINO_CONFIG, fold=0, sync_fn=sync_best_to_drive)

In [ ]:
# 8-1. DINOv2 Finetune Fold 1
run_finetune_fold(DINO_CONFIG, fold=1, sync_fn=sync_best_to_drive)

In [ ]:
# 8-2. DINOv2 Finetune Fold 2
run_finetune_fold(DINO_CONFIG, fold=2, sync_fn=sync_best_to_drive)

In [ ]:
# 8-3. DINOv2 Finetune Fold 3
run_finetune_fold(DINO_CONFIG, fold=3, sync_fn=sync_best_to_drive)

In [ ]:
# 8-4. DINOv2 Finetune Fold 4
run_finetune_fold(DINO_CONFIG, fold=4, sync_fn=sync_best_to_drive)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 9. DINOv2 학습 곡선 분석 + 공통 시각화 함수
# ═══════════════════════════════════════════════════════════════════
import os, glob
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import numpy as np

# 한글 폰트
import subprocess, sys
_noto = "/usr/share/fonts/truetype/noto/NotoSansKR-Regular.otf"
if not os.path.exists(_noto):
    subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-noto-cjk"], check=False)
    fm._load_fontmanager(try_read_cache=False)
for f in fm.findSystemFonts():
    if "NotoSansCJK" in f or "NotoSansKR" in f:
        fm.fontManager.addfont(f)
        plt.rcParams["font.family"] = fm.FontProperties(fname=f).get_name()
        break
plt.rcParams["axes.unicode_minus"] = False

os.chdir("/content/dacon")

def plot_learning_curves(backbone, title_suffix=""):
    """학습 곡선 + 과적합 분석을 시각화하는 공통 함수"""
    ckpt_dir = "checkpoints"
    log_files = sorted(glob.glob(os.path.join(ckpt_dir, f"{backbone}_fold*_log.csv")))
    if not log_files:
        print(f"[WARN] {backbone} 로그 파일이 없습니다. 학습을 먼저 실행하세요.")
        return []

    n_folds = len(log_files)
    fig, axes = plt.subplots(2, n_folds, figsize=(5 * n_folds, 8), squeeze=False)
    fig.suptitle(f"{backbone} 학습 곡선 분석 {title_suffix}", fontsize=14, fontweight="bold")

    all_summaries = []
    for i, log_path in enumerate(log_files):
        df = pd.read_csv(log_path)
        fold_name = os.path.basename(log_path).replace("_log.csv", "")
        epochs = df["epoch"].values
        train_loss = df["train_loss"].values
        val_loss = df["val_logloss"].values

        ax1 = axes[0][i]
        ax1.plot(epochs, train_loss, 'b-o', markersize=3, label="Train Loss", linewidth=1.5)
        ax1.plot(epochs, val_loss, 'r-s', markersize=3, label="Val LogLoss", linewidth=1.5)
        best_ep_idx = val_loss.argmin()
        best_ep = epochs[best_ep_idx]
        best_val = val_loss[best_ep_idx]
        ax1.axvline(best_ep, color='green', linestyle='--', alpha=0.7, label=f"Best ep={best_ep}")
        ax1.scatter([best_ep], [best_val], color='green', s=100, zorder=5, marker='*')
        ax1.set_title(f"{fold_name}", fontsize=10)
        ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
        ax1.legend(fontsize=7); ax1.set_yscale("log"); ax1.grid(True, alpha=0.3)

        ax2 = axes[1][i]
        gap = val_loss - train_loss
        gap_ratio = val_loss / np.clip(train_loss, 1e-8, None)
        ax2.bar(epochs, gap,
                color=['green' if g < 0.02 else 'orange' if g < 0.05 else 'red' for g in gap],
                alpha=0.7, edgecolor='gray', linewidth=0.5)
        ax2.axhline(0, color='black', linewidth=0.5)
        ax2.axhline(0.02, color='green', linestyle=':', alpha=0.5, label="양호 (0.02)")
        ax2.axhline(0.05, color='orange', linestyle=':', alpha=0.5, label="경고 (0.05)")
        ax2.set_xlabel("Epoch"); ax2.set_ylabel("Val - Train Gap")
        ax2.set_title("과적합 갭", fontsize=9); ax2.legend(fontsize=7); ax2.grid(True, alpha=0.3)

        all_summaries.append({
            "fold": fold_name, "total_epochs": len(epochs),
            "best_val_ep": best_ep, "best_val_loss": best_val,
            "best_train_loss": train_loss[best_ep_idx],
            "gap_at_best": gap[best_ep_idx], "gap_ratio_at_best": gap_ratio[best_ep_idx],
        })

    plt.tight_layout(); plt.show()

    print(f"\n{'='*90}")
    print(f"  {'Fold':<30} {'Best Ep':>8} {'Val Loss':>12} {'Train Loss':>12} {'Gap':>10} {'Gap배율':>8}")
    print(f"{'='*90}")
    for s in all_summaries:
        gap_status = "OK" if s["gap_at_best"] < 0.02 else "WARN" if s["gap_at_best"] < 0.05 else "OVERFIT"
        print(f"  {s['fold']:<30} {s['best_val_ep']:>8} {s['best_val_loss']:>12.7f} "
              f"{s['best_train_loss']:>12.7f} {s['gap_at_best']:>10.4f} "
              f"{s['gap_ratio_at_best']:>7.1f}x  {gap_status}")

    avg_gap = np.mean([s["gap_at_best"] for s in all_summaries])
    print(f"\n  평균 과적합 갭: {avg_gap:.4f}")
    if avg_gap > 0.05:
        print("  [!] 과적합 심각 -> BB_LR 더 낮추거나 regularization 강화 필요")
    elif avg_gap > 0.02:
        print("  [~] 경미한 과적합 -> 현재 설정 유지 가능, 모니터링 필요")
    else:
        print("  [v] 과적합 양호 -> 현재 설정 적절")
    return all_summaries

# --- DINOv2 학습 곡선 ---
dino_summaries = plot_learning_curves("dinov2_large", "(Stage 1: 구조물 인식)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 10. EVA-Giant 학습 설정 (Stage 2: 안정성 분석 전문가)
# ═══════════════════════════════════════════════════════════════════
# EVA-Giant: 1.0B params → 과적합 위험 높음
# 매우 보수적 BB_LR (3e-6) + 강한 규제 (WD=0.08, Drop=0.4)
# ═══════════════════════════════════════════════════════════════════
import os
os.chdir("/content/dacon")

EVA_CONFIG = {
    "backbone": "eva_giant",
    "epochs": 15,
    "patience": 8,
    "head_lr": 3e-5,
    "bb_lr": 3e-6,          # DINOv2 대비 3배 낮음 (과적합 억제)
    "weight_decay": 0.08,    # 더 강한 규제
    "drop_rate": 0.4,        # 더 강한 dropout
    "layer_decay": 0.9,      # EVA-Giant 권장값
    "warmup_epochs": 2,
    "merge_dev": True,
    "loss": "ce",
    "no_mixup": True,
    "simple_aug": True,
    "head_type": "simple",
    "scheduler": "cosine",
    "batch_size": 16,
}

print(f"{'='*60}")
print(f"  EVA-Giant 학습 설정 (안정성 분석 전문가)")
print(f"{'='*60}")
for k, v in EVA_CONFIG.items():
    print(f"  {k:>16}: {v}")

print(f"\n  [특성] EVA-Giant는 fine-grained classification 강함")
print(f"  → 안정/불안정 세밀 분류에 강하나 배경 의존 위험")
print(f"  → 보수적 BB_LR + 강한 규제로 배경 과적합 억제")

print(f"\n{'='*60}")
print(f"  DINOv2 vs EVA 설정 비교")
print(f"{'='*60}")
print(f"  {'설정':<16} {'DINOv2':>12} {'EVA':>12}")
print(f"  {'-'*42}")
for k in ["bb_lr", "head_lr", "weight_decay", "drop_rate", "layer_decay", "epochs", "patience"]:
    print(f"  {k:<16} {str(DINO_CONFIG[k]):>12} {str(EVA_CONFIG[k]):>12}")

for fold in range(5):
    best = os.path.join("checkpoints", f"eva_giant_fold{fold}.pth")
    status = "✓ 완료" if os.path.exists(best) else "✗ 미완료"
    print(f"  Fold {fold}: {status}")

In [ ]:
# 11-0. EVA-Giant Finetune Fold 0
run_finetune_fold(EVA_CONFIG, fold=0, sync_fn=sync_best_to_drive)

In [ ]:
# 11-1. EVA-Giant Finetune Fold 1
run_finetune_fold(EVA_CONFIG, fold=1, sync_fn=sync_best_to_drive)

In [ ]:
# 11-2. EVA-Giant Finetune Fold 2
run_finetune_fold(EVA_CONFIG, fold=2, sync_fn=sync_best_to_drive)

In [ ]:
# 11-3. EVA-Giant Finetune Fold 3
run_finetune_fold(EVA_CONFIG, fold=3, sync_fn=sync_best_to_drive)

In [ ]:
# 11-4. EVA-Giant Finetune Fold 4
run_finetune_fold(EVA_CONFIG, fold=4, sync_fn=sync_best_to_drive)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 12. EVA-Giant 학습 곡선 분석
# ═══════════════════════════════════════════════════════════════════
eva_summaries = plot_learning_curves("eva_giant", "(Stage 2: 안정성 분석)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 13. 양쪽 모델 Train/Dev Loss 비교 (DINOv2 vs EVA-Giant)
# ═══════════════════════════════════════════════════════════════════
import torch, os, cv2, numpy as np, pandas as pd
import torch.nn.functional as TF
from torch.amp import autocast
from sklearn.metrics import log_loss as sk_logloss
import albumentations as A
from albumentations.pytorch import ToTensorV2

os.chdir("/content/dacon")
from models import build_model, get_backbone_config

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_split(csv_path, data_dir, img_size, crop_ratio=0.9):
    """CSV + 이미지 → (front_tensors, top_tensors, labels)"""
    df = pd.read_csv(csv_path)
    norm = dict(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    crop = int(img_size * crop_ratio)
    tf = A.Compose([
        A.CenterCrop(crop, crop), A.Resize(img_size, img_size),
        A.Normalize(**norm), ToTensorV2(),
    ])
    fronts, tops, labels = [], [], []
    for _, row in df.iterrows():
        sid = row["id"]
        d = os.path.join(data_dir, sid)
        fr = cv2.cvtColor(cv2.imread(os.path.join(d, "front.png")), cv2.COLOR_BGR2RGB)
        tp = cv2.cvtColor(cv2.imread(os.path.join(d, "top.png")), cv2.COLOR_BGR2RGB)
        fronts.append(tf(image=fr)["image"])
        tops.append(tf(image=tp)["image"])
        labels.append(1 if row["label"] == "unstable" else 0)
    return torch.stack(fronts), torch.stack(tops), np.array(labels)

def evaluate_model(model, fronts, tops, labels, device, batch_size=32):
    """모델의 logloss, accuracy, per-class accuracy 계산"""
    model.eval()
    all_probs = []
    n = len(labels)
    with torch.no_grad():
        for i in range(0, n, batch_size):
            fr = fronts[i:i+batch_size].to(device)
            tp = tops[i:i+batch_size].to(device)
            with autocast("cuda"):
                logits = model(fr, tp)
            probs = TF.softmax(logits.float(), dim=1).cpu().numpy()
            all_probs.append(probs)
    all_probs = np.concatenate(all_probs)
    eps = 1e-7
    clipped = np.clip(all_probs, eps, 1 - eps)
    logloss = sk_logloss(labels, clipped, labels=[0, 1])
    preds = all_probs.argmax(axis=1)
    acc = (preds == labels).mean()
    unstable_mask = labels == 1
    stable_mask = labels == 0
    unstable_acc = (preds[unstable_mask] == 1).mean() if unstable_mask.sum() > 0 else 0
    stable_acc = (preds[stable_mask] == 0).mean() if stable_mask.sum() > 0 else 0
    return {
        "logloss": logloss, "accuracy": acc,
        "unstable_acc": unstable_acc, "stable_acc": stable_acc,
        "probs": all_probs,
    }

# --- 데이터 로드 (336px - 양쪽 동일 img_size) ---
img_size = 336
print("Loading train data...")
train_fronts, train_tops, train_labels = load_split(
    "data/open/train.csv", "data/open/train", img_size)
print(f"  Train: {len(train_labels)} (unstable={train_labels.sum()}, stable={(1-train_labels).sum()})")

print("Loading dev data...")
dev_fronts, dev_tops, dev_labels = load_split(
    "data/open/dev.csv", "data/open/dev", img_size)
print(f"  Dev: {len(dev_labels)} (unstable={dev_labels.sum()}, stable={(1-dev_labels).sum()})")

# --- 모든 체크포인트 평가 ---
ckpt_dir = "checkpoints"
results = []
for bb in ["dinov2_large", "eva_giant"]:
    ht = HEAD_TYPE_MAP.get(bb, "simple")  # 백본별 올바른 head type 사용
    all_ckpts = sorted([f for f in os.listdir(ckpt_dir)
                        if f.startswith(bb) and f.endswith('.pth')
                        and 'ckpt' not in f and 'pretrain' not in f])
    for ckpt_name in all_ckpts:
        ckpt_path = os.path.join(ckpt_dir, ckpt_name)
        model = build_model(bb, pretrained=False, num_classes=2, drop_rate=0.0, head_type=ht)
        sd = torch.load(ckpt_path, map_location="cpu", weights_only=True)
        model_sd = model.state_dict()
        filtered = {k: v for k, v in sd.items() if k in model_sd and v.shape == model_sd[k].shape}
        model.load_state_dict(filtered, strict=False)
        model = model.to(device).eval()

        train_res = evaluate_model(model, train_fronts, train_tops, train_labels, device)
        dev_res = evaluate_model(model, dev_fronts, dev_tops, dev_labels, device)

        results.append({
            "backbone": bb, "checkpoint": ckpt_name,
            "train_ll": train_res["logloss"], "dev_ll": dev_res["logloss"],
            "train_acc": train_res["accuracy"], "dev_acc": dev_res["accuracy"],
            "train_unstable": train_res["unstable_acc"], "train_stable": train_res["stable_acc"],
            "dev_unstable": dev_res["unstable_acc"], "dev_stable": dev_res["stable_acc"],
        })
        print(f"  {ckpt_name:<35} T:{train_res['logloss']:.6f} D:{dev_res['logloss']:.6f} "
              f"D-Acc:{dev_res['accuracy']:.4f} D-Unst:{dev_res['unstable_acc']:.4f} D-Stab:{dev_res['stable_acc']:.4f}")
        del model; torch.cuda.empty_cache()

# --- 비교 요약 ---
print(f"\n{'='*110}")
print(f"  {'Checkpoint':<35} {'Train LL':>11} {'Dev LL':>11} {'D-Acc':>7} {'D-Unst':>7} {'D-Stab':>7}")
print(f"  {'-'*80}")
for r in results:
    print(f"  {r['checkpoint']:<35} {r['train_ll']:>11.7f} {r['dev_ll']:>11.7f} "
          f"{r['dev_acc']:>6.4f} {r['dev_unstable']:>6.4f} {r['dev_stable']:>6.4f}")

for bb in ["dinov2_large", "eva_giant"]:
    bb_results = [r for r in results if r["backbone"] == bb]
    if bb_results:
        avg_dev = np.mean([r["dev_ll"] for r in bb_results])
        avg_unstable = np.mean([r["dev_unstable"] for r in bb_results])
        avg_stable = np.mean([r["dev_stable"] for r in bb_results])
        print(f"\n  [{bb}] 평균: Dev LL={avg_dev:.6f} Unstable Acc={avg_unstable:.4f} Stable Acc={avg_stable:.4f}")

print(f"\n  [분석] DINOv2 → Stable Acc 높음 (구조물 인식), EVA → Unstable Acc 높음 (세부 분석)")
print(f"  [결론] 상보적 결합으로 양쪽 강점 활용 가능")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 14. Stacking Meta-Learner (OOF 예측 + 최적 가중치 탐색)
# ═══════════════════════════════════════════════════════════════════
# Out-Of-Fold 예측으로 DINOv2와 EVA의 최적 결합 계수를 학습
# ═══════════════════════════════════════════════════════════════════
import os, pickle
import numpy as np, pandas as pd, torch
import torch.nn.functional as TF
from torch.amp import autocast
from sklearn.metrics import log_loss as sk_logloss
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

os.chdir("/content/dacon")
from models import build_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- OOF 예측 생성 ---
# MERGE_DEV=True이면 전체 데이터(train+dev)를 5-Fold로 나눈 것이므로
# 각 fold의 validation set이 OOF가 됨
train_df = pd.read_csv("data/open/train.csv")
dev_df = pd.read_csv("data/open/dev.csv")
all_df = pd.concat([train_df, dev_df], ignore_index=True)
all_labels = np.array([1 if l == "unstable" else 0 for l in all_df["label"]])
n_total = len(all_df)
print(f"전체 데이터: {n_total}개 (unstable={all_labels.sum()}, stable={(1-all_labels).sum()})")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = list(skf.split(np.zeros(n_total), all_labels))

import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2

img_size = 336
norm = dict(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
crop = int(img_size * 0.9)
val_tf = A.Compose([A.CenterCrop(crop, crop), A.Resize(img_size, img_size),
                     A.Normalize(**norm), ToTensorV2()])

def get_oof_preds(backbone, head_type="simple"):
    """각 fold의 validation set 예측을 모아 전체 OOF 생성"""
    oof_probs = np.zeros((n_total, 2))
    oof_valid = np.zeros(n_total, dtype=bool)

    for fold_idx, (train_idx, val_idx) in enumerate(folds):
        ckpt_path = os.path.join("checkpoints", f"{backbone}_fold{fold_idx}.pth")
        if not os.path.exists(ckpt_path):
            print(f"  [SKIP] {backbone} fold {fold_idx} 체크포인트 없음")
            continue

        model = build_model(backbone, pretrained=False, num_classes=2,
                           drop_rate=0.0, head_type=head_type)
        sd = torch.load(ckpt_path, map_location="cpu", weights_only=True)
        model_sd = model.state_dict()
        filtered = {k: v for k, v in sd.items() if k in model_sd and v.shape == model_sd[k].shape}
        model.load_state_dict(filtered, strict=False)
        model = model.to(device).eval()

        val_probs = []
        with torch.no_grad():
            for i in range(0, len(val_idx), 32):
                batch_idx = val_idx[i:i+32]
                fronts, tops = [], []
                for idx in batch_idx:
                    row = all_df.iloc[idx]
                    sid = row["id"]
                    split_dir = "data/open/train" if idx < len(train_df) else "data/open/dev"
                    d = os.path.join(split_dir, sid)
                    fr = cv2.cvtColor(cv2.imread(os.path.join(d, "front.png")), cv2.COLOR_BGR2RGB)
                    tp = cv2.cvtColor(cv2.imread(os.path.join(d, "top.png")), cv2.COLOR_BGR2RGB)
                    fronts.append(val_tf(image=fr)["image"])
                    tops.append(val_tf(image=tp)["image"])
                fr_t = torch.stack(fronts).to(device)
                tp_t = torch.stack(tops).to(device)
                with autocast("cuda"):
                    logits = model(fr_t, tp_t)
                probs = TF.softmax(logits.float(), dim=1).cpu().numpy()
                val_probs.append(probs)

        val_probs = np.concatenate(val_probs)
        oof_probs[val_idx] = val_probs
        oof_valid[val_idx] = True

        fold_ll = sk_logloss(all_labels[val_idx], np.clip(val_probs, 1e-7, 1-1e-7), labels=[0, 1])
        print(f"  {backbone} fold {fold_idx}: OOF logloss={fold_ll:.6f} ({len(val_idx)} samples)")
        del model; torch.cuda.empty_cache()

    return oof_probs, oof_valid

print("DINOv2 OOF 예측 생성...")
dino_oof, dino_valid = get_oof_preds("dinov2_large", head_type=HEAD_TYPE_MAP["dinov2_large"])
print("\nEVA-Giant OOF 예측 생성...")
eva_oof, eva_valid = get_oof_preds("eva_giant", head_type=HEAD_TYPE_MAP["eva_giant"])

# 양쪽 모두 유효한 샘플만 사용
both_valid = dino_valid & eva_valid
valid_labels = all_labels[both_valid]
dino_oof_valid = dino_oof[both_valid]
eva_oof_valid = eva_oof[both_valid]
print(f"\n양쪽 유효 샘플: {both_valid.sum()}/{n_total}")

eps = 1e-7
dino_ll = sk_logloss(valid_labels, np.clip(dino_oof_valid, eps, 1-eps), labels=[0, 1])
eva_ll = sk_logloss(valid_labels, np.clip(eva_oof_valid, eps, 1-eps), labels=[0, 1])
print(f"\n  DINOv2 단독 OOF LogLoss: {dino_ll:.6f}")
print(f"  EVA    단독 OOF LogLoss: {eva_ll:.6f}")

# --- 최적 가중치 그리드 서치 ---
best_w, best_ll = 0, float("inf")
print(f"\n가중치 그리드 서치: w * DINOv2 + (1-w) * EVA")
for w in np.arange(0.0, 1.01, 0.05):
    ens = w * dino_oof_valid + (1 - w) * eva_oof_valid
    ll = sk_logloss(valid_labels, np.clip(ens, eps, 1-eps), labels=[0, 1])
    marker = " <<<" if ll < best_ll else ""
    if ll < best_ll:
        best_w, best_ll = w, ll
    if abs(w % 0.1) < 0.01 or marker:
        print(f"  w={w:.2f}: logloss={ll:.6f}{marker}")

print(f"\n  ★ 최적 가중치: w={best_w:.2f} (logloss={best_ll:.6f})")
print(f"  개선: DINOv2 대비 {dino_ll - best_ll:+.6f}, EVA 대비 {eva_ll - best_ll:+.6f}")

# --- Logistic Regression 스태킹 ---
X_stack = np.column_stack([
    dino_oof_valid[:, 0], dino_oof_valid[:, 1],
    eva_oof_valid[:, 0], eva_oof_valid[:, 1],
])
lr_model = LogisticRegression(C=1.0, solver="lbfgs", max_iter=1000)
lr_model.fit(X_stack, valid_labels)
lr_probs = lr_model.predict_proba(X_stack)
lr_ll = sk_logloss(valid_labels, np.clip(lr_probs, eps, 1-eps), labels=[0, 1])
print(f"  LR Stacking OOF LogLoss: {lr_ll:.6f}")

# --- XGBoost 스태킹 (옵션) ---
try:
    from xgboost import XGBClassifier
    xgb_model = XGBClassifier(
        n_estimators=100, max_depth=3, learning_rate=0.1,
        eval_metric="logloss", use_label_encoder=False, verbosity=0
    )
    xgb_model.fit(X_stack, valid_labels)
    xgb_probs = xgb_model.predict_proba(X_stack)
    xgb_ll = sk_logloss(valid_labels, np.clip(xgb_probs, eps, 1-eps), labels=[0, 1])
    print(f"  XGB Stacking OOF LogLoss: {xgb_ll:.6f}")
except Exception as e:
    xgb_model = None
    print(f"  XGB Stacking: 실패 ({e})")

# --- 결과 저장 ---
OPTIMAL_W = best_w
META_LR = lr_model
META_XGB = xgb_model if 'xgb_model' in dir() else None

# 메타러너 저장
meta_path = os.path.join("checkpoints", "meta_learner.pkl")
with open(meta_path, "wb") as f:
    pickle.dump({"optimal_w": OPTIMAL_W, "lr_model": lr_model,
                 "xgb_model": META_XGB}, f)
print(f"\n메타러너 저장: {meta_path}")

print(f"\n{'='*60}")
print(f"  최종 결과 요약")
print(f"{'='*60}")
print(f"  DINOv2 단독:     {dino_ll:.6f}")
print(f"  EVA 단독:        {eva_ll:.6f}")
print(f"  가중 평균 (w={best_w:.2f}): {best_ll:.6f}")
print(f"  LR Stacking:     {lr_ll:.6f}")
if META_XGB:
    print(f"  XGB Stacking:    {xgb_ll:.6f}")
print(f"{'='*60}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 15. 2-Stage Ensemble 추론 (DINOv2 + EVA-Giant → 테스트셋)
# ═══════════════════════════════════════════════════════════════════
import os, glob, datetime, pickle
import numpy as np, pandas as pd, torch
import torch.nn.functional as TF
from torch.amp import autocast
from torch.utils.data import DataLoader

os.chdir("/content/dacon")
from models import build_model, get_backbone_config
from inference_v2 import (
    DualCropDataset, make_dual_transforms, make_dual_tta_transforms,
    predict, predict_tta,
)

# ═══════════════════ CONFIG ═══════════════════
USE_FOLDS = [0, 1, 2, 3, 4]
USE_TTA = True
FRONT_CROP = 0.9
TOP_CROP = 0.9
NUM_WORKERS = 0
# ══════════════════════════════════════════════

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
test_csv = "data/open/sample_submission.csv"
test_dir = "data/open/test"

def run_model_inference(backbone):
    """단일 백본의 5-Fold TTA 추론 → 평균 확률"""
    ht = HEAD_TYPE_MAP.get(backbone, "simple")  # 백본별 올바른 head type 사용
    cfg = get_backbone_config(backbone)
    img_size = cfg["img_size"]
    bs = 32

    all_preds = []
    for fold in USE_FOLDS:
        ckpt_path = os.path.join("checkpoints", f"{backbone}_fold{fold}.pth")
        if not os.path.exists(ckpt_path):
            print(f"  [SKIP] {ckpt_path}")
            continue

        model = build_model(backbone, pretrained=False, num_classes=2,
                           drop_rate=0.0, head_type=ht)
        sd = torch.load(ckpt_path, map_location=device, weights_only=True)
        model_sd = model.state_dict()
        filtered = {k: v for k, v in sd.items() if k in model_sd and v.shape == model_sd[k].shape}
        model.load_state_dict(filtered, strict=False)
        model = model.to(device)

        if USE_TTA:
            probs, ids = predict_tta(model, test_csv, test_dir, img_size, device,
                                     FRONT_CROP, TOP_CROP, bs=bs, nw=NUM_WORKERS)
        else:
            ftf, ttf = make_dual_transforms(img_size, FRONT_CROP, TOP_CROP)
            ds = DualCropDataset(test_csv, test_dir, ftf, ttf)
            loader = DataLoader(ds, batch_size=bs, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=True)
            probs, ids = predict(model, loader, device)

        all_preds.append(probs)
        print(f"  {backbone} fold {fold}: P(unstable) mean={probs[:, 1].mean():.6f}")
        del model; torch.cuda.empty_cache()

    if not all_preds:
        return None, None
    mean_probs = np.mean(all_preds, axis=0)
    return mean_probs, ids

# --- DINOv2 추론 ---
print(f"{'='*60}")
print(f"Stage 1: DINOv2-Large 추론 {'(TTA)' if USE_TTA else ''}")
print(f"{'='*60}")
dino_test_probs, test_ids = run_model_inference("dinov2_large")

# --- EVA-Giant 추론 ---
print(f"\n{'='*60}")
print(f"Stage 2: EVA-Giant 추론 {'(TTA)' if USE_TTA else ''}")
print(f"{'='*60}")
eva_test_probs, _ = run_model_inference("eva_giant")

# --- 확률 분포 출력 ---
def print_dist(name, probs):
    p = probs[:, 1]
    print(f"\n  [{name}]")
    print(f"    mean={p.mean():.8f} std={p.std():.8f} min={p.min():.8f} max={p.max():.8f}")
    print(f"    unstable(>0.5): {(p > 0.5).sum()} | stable(≤0.5): {(p <= 0.5).sum()}")

if dino_test_probs is not None:
    print_dist("DINOv2", dino_test_probs)
if eva_test_probs is not None:
    print_dist("EVA-Giant", eva_test_probs)

# --- 결합 ---
if dino_test_probs is not None and eva_test_probs is not None:
    # 메타러너 로드
    meta_path = os.path.join("checkpoints", "meta_learner.pkl")
    if os.path.exists(meta_path):
        with open(meta_path, "rb") as f:
            meta = pickle.load(f)
        OPTIMAL_W = meta["optimal_w"]
        META_LR = meta["lr_model"]
        META_XGB = meta.get("xgb_model")
    else:
        print("[WARN] 메타러너 없음 → 기본 w=0.3 사용")
        OPTIMAL_W = 0.3
        META_LR = None
        META_XGB = None

    # 1) 가중 평균
    weighted_probs = OPTIMAL_W * dino_test_probs + (1 - OPTIMAL_W) * eva_test_probs
    print_dist(f"가중 평균 (w={OPTIMAL_W:.2f})", weighted_probs)

    # 2) LR Stacking
    if META_LR is not None:
        X_test = np.column_stack([
            dino_test_probs[:, 0], dino_test_probs[:, 1],
            eva_test_probs[:, 0], eva_test_probs[:, 1],
        ])
        lr_probs = META_LR.predict_proba(X_test)
        print_dist("LR Stacking", lr_probs)

    # 3) XGB Stacking
    if META_XGB is not None:
        xgb_probs = META_XGB.predict_proba(X_test)
        print_dist("XGB Stacking", xgb_probs)
else:
    print("\n[WARN] 한쪽 모델 체크포인트 부족 - 단독 추론만 가능")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 16. 앙상블 가중치 최적화 (Dev/OOF 기반 상세 분석)
# ═══════════════════════════════════════════════════════════════════
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import log_loss as sk_logloss

# OOF 데이터가 위 셀에서 생성되었다고 가정
# dino_oof_valid, eva_oof_valid, valid_labels 사용

eps = 1e-7

# --- 세밀한 가중치 탐색 (0.01 단위) ---
alphas = np.linspace(0, 1, 101)
losses = []
for a in alphas:
    ens = a * dino_oof_valid + (1 - a) * eva_oof_valid
    ll = sk_logloss(valid_labels, np.clip(ens, eps, 1-eps), labels=[0, 1])
    losses.append(ll)
losses = np.array(losses)

best_idx = losses.argmin()
best_alpha = alphas[best_idx]
best_loss = losses[best_idx]

# --- 시각화 ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1) Alpha vs LogLoss
ax = axes[0]
ax.plot(alphas, losses, 'b-', linewidth=2)
ax.axvline(best_alpha, color='red', linestyle='--', label=f"최적 α={best_alpha:.2f}")
ax.scatter([best_alpha], [best_loss], color='red', s=100, zorder=5, marker='*')
ax.axhline(sk_logloss(valid_labels, np.clip(dino_oof_valid, eps, 1-eps), labels=[0,1]),
           color='green', linestyle=':', alpha=0.7, label="DINOv2 단독")
ax.axhline(sk_logloss(valid_labels, np.clip(eva_oof_valid, eps, 1-eps), labels=[0,1]),
           color='orange', linestyle=':', alpha=0.7, label="EVA 단독")
ax.set_xlabel("α (DINOv2 가중치)")
ax.set_ylabel("OOF LogLoss")
ax.set_title("Ensemble Weight Optimization\nα · DINOv2 + (1-α) · EVA")
ax.legend()
ax.grid(True, alpha=0.3)

# 2) 신뢰도 기반 라우팅 분석
dino_conf = np.abs(dino_oof_valid[:, 1] - 0.5) * 2  # 0~1 범위 신뢰도
eva_conf = np.abs(eva_oof_valid[:, 1] - 0.5) * 2

ax = axes[1]
dino_correct = (dino_oof_valid.argmax(1) == valid_labels)
eva_correct = (eva_oof_valid.argmax(1) == valid_labels)
colors = np.where(dino_correct & eva_correct, 'green',
         np.where(dino_correct & ~eva_correct, 'blue',
         np.where(~dino_correct & eva_correct, 'orange', 'red')))
ax.scatter(dino_conf, eva_conf, c=colors, alpha=0.5, s=20, edgecolors='none')
ax.set_xlabel("DINOv2 신뢰도")
ax.set_ylabel("EVA 신뢰도")
ax.set_title("신뢰도 분포 (정답 기준)")
# 범례
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='green', markersize=8, label='양쪽 정답'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', markersize=8, label='DINO만 정답'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='orange', markersize=8, label='EVA만 정답'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=8, label='양쪽 오답'),
]
ax.legend(handles=legend_elements, loc='lower right')
ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

# --- 통계 ---
both_right = (dino_correct & eva_correct).sum()
only_dino = (dino_correct & ~eva_correct).sum()
only_eva = (~dino_correct & eva_correct).sum()
both_wrong = (~dino_correct & ~eva_correct).sum()

print(f"\n{'='*50}")
print(f"  상보성 분석 (OOF 기준)")
print(f"{'='*50}")
print(f"  양쪽 정답:   {both_right:4d} ({both_right/len(valid_labels):.1%})")
print(f"  DINO만 정답: {only_dino:4d} ({only_dino/len(valid_labels):.1%})")
print(f"  EVA만 정답:  {only_eva:4d} ({only_eva/len(valid_labels):.1%})")
print(f"  양쪽 오답:   {both_wrong:4d} ({both_wrong/len(valid_labels):.1%})")
print(f"\n  ★ 최적 앙상블 가중치: α={best_alpha:.2f} → LogLoss={best_loss:.6f}")
print(f"  결합으로 복구 가능한 오류: {only_dino + only_eva}개 / {only_dino + only_eva + both_wrong}개")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 17. 제출 파일 생성 + 검증 (가중평균 / LR Stacking / XGB)
# ═══════════════════════════════════════════════════════════════════
import os, datetime
import numpy as np, pandas as pd

os.chdir("/content/dacon")
out_dir = "submissions"
os.makedirs(out_dir, exist_ok=True)

sub_df = pd.read_csv("data/open/sample_submission.csv", encoding="utf-8-sig")
ts = datetime.datetime.now().strftime("%m%d_%H%M")
eps = 1e-7

submissions = {}

# 1) 가중 평균
if dino_test_probs is not None and eva_test_probs is not None:
    w_probs = OPTIMAL_W * dino_test_probs + (1 - OPTIMAL_W) * eva_test_probs
    submissions[f"dino_eva_w{OPTIMAL_W:.0%}"] = w_probs

# 2) LR Stacking
if META_LR is not None and dino_test_probs is not None and eva_test_probs is not None:
    X_t = np.column_stack([dino_test_probs[:, 0], dino_test_probs[:, 1],
                           eva_test_probs[:, 0], eva_test_probs[:, 1]])
    submissions["dino_eva_lr_stack"] = META_LR.predict_proba(X_t)

# 3) XGB Stacking
if META_XGB is not None and dino_test_probs is not None and eva_test_probs is not None:
    submissions["dino_eva_xgb_stack"] = META_XGB.predict_proba(X_t)

# 4) 단독 모델 (비교용)
if dino_test_probs is not None:
    submissions["dinov2_5fold"] = dino_test_probs
if eva_test_probs is not None:
    submissions["eva_giant_5fold"] = eva_test_probs

tta_tag = "_tta" if USE_TTA else ""

print(f"{'='*60}")
print(f"  제출 파일 생성")
print(f"{'='*60}")

for name, probs in submissions.items():
    sub = sub_df.copy()
    sub["unstable_prob"] = np.clip(probs[:, 1], eps, 1 - eps)
    sub["stable_prob"] = 1.0 - sub["unstable_prob"]

    fname = f"submission_{name}{tta_tag}_{ts}.csv"
    fpath = os.path.join(out_dir, fname)
    sub[["id", "unstable_prob", "stable_prob"]].to_csv(fpath, index=False)

    p = sub["unstable_prob"]
    n_unstable = (p > 0.5).sum()
    print(f"\n  [{name}]")
    print(f"    파일: {fname}")
    print(f"    mean={p.mean():.8f}  std={p.std():.8f}")
    print(f"    min={p.min():.8f}  max={p.max():.8f}")
    print(f"    unstable: {n_unstable} | stable: {len(p) - n_unstable}")

# --- 검증 ---
print(f"\n{'='*60}")
print(f"  검증")
print(f"{'='*60}")
for name, probs in submissions.items():
    p = probs
    assert not np.isnan(p).any(), f"{name}: NaN 존재!"
    assert np.allclose(p.sum(axis=1), 1.0, atol=1e-5), f"{name}: 확률 합 != 1"
    print(f"  [OK] {name}: NaN 없음, 확률 합 ≈ 1.0")

print(f"\n  총 {len(submissions)}개 제출 파일 생성 완료")
print(f"  저장 위치: {out_dir}/ → Drive 동기화됨")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 18. 모델별 vs 앙상블 성능 비교 시각화
# ═══════════════════════════════════════════════════════════════════
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import log_loss as sk_logloss

eps = 1e-7

# OOF 데이터 사용 (stacking 셀에서 생성)
d_p = dino_oof_valid[:, 1]
e_p = eva_oof_valid[:, 1]
ens_p = (OPTIMAL_W * dino_oof_valid + (1 - OPTIMAL_W) * eva_oof_valid)[:, 1]

fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# 1) 히스토그램 오버레이
ax = axes[0][0]
bins = np.linspace(0, 1, 51)
ax.hist(d_p, bins=bins, alpha=0.4, label="DINOv2", color="blue", density=True)
ax.hist(e_p, bins=bins, alpha=0.4, label="EVA-Giant", color="orange", density=True)
ax.hist(ens_p, bins=bins, alpha=0.5, label=f"앙상블 (α={OPTIMAL_W:.2f})", color="green", density=True)
ax.axvline(0.5, color='red', linestyle='--', alpha=0.7)
ax.set_xlabel("P(unstable)"); ax.set_ylabel("Density")
ax.set_title("OOF 확률 분포 비교"); ax.legend(); ax.grid(True, alpha=0.3)

# 2) 산점도: DINOv2 vs EVA
ax = axes[0][1]
colors = np.where(valid_labels == 1, 'red', 'blue')
ax.scatter(d_p, e_p, c=colors, alpha=0.4, s=15, edgecolors='none')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax.axvline(0.5, color='gray', linestyle=':', alpha=0.5)
ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel("DINOv2 P(unstable)"); ax.set_ylabel("EVA P(unstable)")
ax.set_title("DINOv2 vs EVA 예측 산점도\n(빨강=unstable, 파랑=stable)")
ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
ax.grid(True, alpha=0.3)

# 3) 메트릭 바 차트
ax = axes[1][0]
models_names = ["DINOv2", "EVA-Giant", f"앙상블\n(α={OPTIMAL_W:.2f})"]
d_preds = (d_p > 0.5).astype(int)
e_preds = (e_p > 0.5).astype(int)
ens_preds = (ens_p > 0.5).astype(int)

unstable_mask = valid_labels == 1
stable_mask = valid_labels == 0

metrics = {
    "LogLoss": [
        sk_logloss(valid_labels, np.clip(dino_oof_valid, eps, 1-eps), labels=[0,1]),
        sk_logloss(valid_labels, np.clip(eva_oof_valid, eps, 1-eps), labels=[0,1]),
        sk_logloss(valid_labels, np.clip(OPTIMAL_W*dino_oof_valid+(1-OPTIMAL_W)*eva_oof_valid, eps, 1-eps), labels=[0,1]),
    ],
    "Accuracy": [
        (d_preds == valid_labels).mean(),
        (e_preds == valid_labels).mean(),
        (ens_preds == valid_labels).mean(),
    ],
    "Unstable\nRecall": [
        (d_preds[unstable_mask] == 1).mean() if unstable_mask.sum() > 0 else 0,
        (e_preds[unstable_mask] == 1).mean() if unstable_mask.sum() > 0 else 0,
        (ens_preds[unstable_mask] == 1).mean() if unstable_mask.sum() > 0 else 0,
    ],
    "Stable\nRecall": [
        (d_preds[stable_mask] == 0).mean() if stable_mask.sum() > 0 else 0,
        (e_preds[stable_mask] == 0).mean() if stable_mask.sum() > 0 else 0,
        (ens_preds[stable_mask] == 0).mean() if stable_mask.sum() > 0 else 0,
    ],
}

x = np.arange(len(models_names))
width = 0.18
for i, (metric_name, values) in enumerate(metrics.items()):
    bars = ax.bar(x + i * width, values, width, label=metric_name, alpha=0.8)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{val:.3f}", ha='center', va='bottom', fontsize=7)
ax.set_xticks(x + 1.5 * width); ax.set_xticklabels(models_names)
ax.set_title("OOF 메트릭 비교"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3, axis='y')

# 4) 상보성 Venn-style bar chart
ax = axes[1][1]
d_correct = (d_preds == valid_labels)
e_correct = (e_preds == valid_labels)
categories = ["양쪽 정답", "DINO만\n정답", "EVA만\n정답", "양쪽 오답"]
counts = [
    (d_correct & e_correct).sum(),
    (d_correct & ~e_correct).sum(),
    (~d_correct & e_correct).sum(),
    (~d_correct & ~e_correct).sum(),
]
colors_bar = ['#2ecc71', '#3498db', '#e67e22', '#e74c3c']
bars = ax.bar(categories, counts, color=colors_bar, alpha=0.8, edgecolor='gray')
for bar, cnt in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f"{cnt}", ha='center', va='bottom', fontweight='bold')
ax.set_ylabel("샘플 수"); ax.set_title("모델 상보성 분석 (OOF)")
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout(); plt.show()

# 숫자 요약
print(f"\n앙상블 효과: {counts[1]+counts[2]}개 상보적 오류 → 결합으로 활용 가능")
print(f"한계: {counts[3]}개 양쪽 모두 틀림 → 추가 모델/데이터 필요")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 19. Attention Rollout 비교 (DINOv2 vs EVA-Giant)
# ═══════════════════════════════════════════════════════════════════
import torch, torch.nn as nn, torch.nn.functional as F
import os, cv2, numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import albumentations as A
from albumentations.pytorch import ToTensorV2

os.chdir("/content/dacon")
from models import build_model, get_backbone_config

N_PER_GROUP = 3
PATCH_SIZE = 14
VIS_FRONT_CROP = 0.9
VIS_TOP_CROP = 0.9
VIS_FOLD = 0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
img_size = 336
grid_size = img_size // PATCH_SIZE
front_crop = int(img_size * VIS_FRONT_CROP)
top_crop = int(img_size * VIS_TOP_CROP)

norm_args = dict(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
front_val_tf = A.Compose([A.CenterCrop(front_crop, front_crop), A.Resize(img_size, img_size),
                           A.Normalize(**norm_args), ToTensorV2()])
top_val_tf = A.Compose([A.CenterCrop(top_crop, top_crop), A.Resize(img_size, img_size),
                         A.Normalize(**norm_args), ToTensorV2()])
front_disp_tf = A.Compose([A.CenterCrop(front_crop, front_crop), A.Resize(img_size, img_size)])
top_disp_tf = A.Compose([A.CenterCrop(top_crop, top_crop), A.Resize(img_size, img_size)])

# --- Attention 추출 (EVA + DINOv2 양쪽 호환) ---
def extract_attentions(backbone, img_tensor):
    """ViT backbone에서 attention weight 추출 (EvaAttention / standard Attention 호환)"""
    attns = []
    hooks = []
    for blk in backbone.blocks:
        am = blk.attn
        def hook_fn(module, inputs, output):
            x = inputs[0]
            B, N, C = x.shape
            if hasattr(module, 'qkv') and module.qkv is not None:
                q_bias = getattr(module, 'q_bias', None)
                if q_bias is not None:
                    k_bias = getattr(module, 'k_bias', torch.zeros_like(q_bias))
                    v_bias = getattr(module, 'v_bias', torch.zeros_like(q_bias))
                    qkv_bias = torch.cat((q_bias, k_bias, v_bias))
                    qkv = F.linear(x, weight=module.qkv.weight, bias=qkv_bias)
                else:
                    qkv = F.linear(x, weight=module.qkv.weight, bias=module.qkv.bias)
                qkv = qkv.reshape(B, N, 3, module.num_heads, -1).permute(2, 0, 3, 1, 4)
                q, k, _ = qkv.unbind(0)
            else:
                q = module.q_proj(x).reshape(B, N, module.num_heads, -1).transpose(1, 2)
                k = module.k_proj(x).reshape(B, N, module.num_heads, -1).transpose(1, 2)
            if hasattr(module, 'q_norm'):
                q = module.q_norm(q)
                k = module.k_norm(k)
            a = (q @ k.transpose(-2, -1)) * module.scale
            attns.append(a.softmax(dim=-1).detach().cpu())
        hooks.append(am.register_forward_hook(hook_fn))
    with torch.no_grad():
        backbone(img_tensor)
    for h in hooks: h.remove()
    return attns

def attention_rollout(attns):
    N = attns[0].shape[-1]
    result = torch.eye(N).unsqueeze(0)
    for attn in attns:
        a = attn.mean(dim=1)
        a = 0.5 * a + 0.5 * torch.eye(N).unsqueeze(0)
        result = a @ result
    cls = result[0, 0, 1:].numpy()
    return cls / (cls.max() + 1e-8)

def attn_to_heatmap(attn_vec, grid_size, img_size):
    h = attn_vec.reshape(grid_size, grid_size)
    h = cv2.resize(h.astype(np.float32), (img_size, img_size), interpolation=cv2.INTER_CUBIC)
    return np.clip(h, 0, 1)

# --- 두 모델 로드 (백본별 올바른 head_type 사용) ---
models_vis = {}
for bb in ["dinov2_large", "eva_giant"]:
    ckpt_path = os.path.join("checkpoints", f"{bb}_fold{VIS_FOLD}.pth")
    if not os.path.exists(ckpt_path):
        print(f"[SKIP] {ckpt_path} 없음")
        continue
    ht = HEAD_TYPE_MAP.get(bb, "simple")
    model = build_model(bb, pretrained=False, num_classes=2, drop_rate=0.0, head_type=ht)
    sd = torch.load(ckpt_path, map_location="cpu", weights_only=True)
    model_sd = model.state_dict()
    filtered = {k: v for k, v in sd.items() if k in model_sd and v.shape == model_sd[k].shape}
    model.load_state_dict(filtered, strict=False)
    model = model.to(device).eval()
    models_vis[bb] = model
    print(f"  [OK] {bb} fold {VIS_FOLD} 로드 (head={ht}, blocks={len(model.backbone.blocks)})")

# --- 테스트 예측으로 샘플 선택 ---
test_dir = os.path.join("data", "open", "test")
sub_df = pd.read_csv(os.path.join("data", "open", "sample_submission.csv"))
all_ids = sub_df["id"].tolist()

test_preds = {}
ref_bb = list(models_vis.keys())[0] if models_vis else None
if ref_bb:
    ref_model = models_vis[ref_bb]
    with torch.no_grad():
        for sid in all_ids:
            d = os.path.join(test_dir, sid)
            fr = cv2.cvtColor(cv2.imread(os.path.join(d, "front.png")), cv2.COLOR_BGR2RGB)
            tp = cv2.cvtColor(cv2.imread(os.path.join(d, "top.png")), cv2.COLOR_BGR2RGB)
            fr_t = front_val_tf(image=fr)["image"].unsqueeze(0).to(device)
            tp_t = top_val_tf(image=tp)["image"].unsqueeze(0).to(device)
            logits = ref_model(fr_t, tp_t)
            test_preds[sid] = float(torch.softmax(logits, dim=1)[0, 1])

sorted_items = sorted(test_preds.items(), key=lambda x: x[1], reverse=True)
top_unstable = [s for s, _ in sorted_items[:N_PER_GROUP]]
top_stable = [s for s, _ in sorted_items[-N_PER_GROUP:]]
mid = len(sorted_items) // 2
borderline = [s for s, _ in sorted_items[mid:mid+N_PER_GROUP]]
groups = [("UNSTABLE", top_unstable), ("BORDERLINE", borderline), ("STABLE", top_stable)]

# --- 시각화 ---
for gname, group_ids in groups:
    n = len(group_ids)
    n_models = len(models_vis)
    ncols = 2 * n_models * n  # front+top per model per sample
    fig, axes = plt.subplots(2, 2 * n, figsize=(4 * n, 7))
    if n == 1: axes = axes.reshape(2, -1)
    fig.suptitle(f"{gname} — DINOv2 vs EVA Attention Rollout", fontsize=13, fontweight="bold", y=1.02)

    for j, sid in enumerate(group_ids):
        d = os.path.join(test_dir, sid)
        fr_raw = cv2.cvtColor(cv2.imread(os.path.join(d, "front.png")), cv2.COLOR_BGR2RGB)
        tp_raw = cv2.cvtColor(cv2.imread(os.path.join(d, "top.png")), cv2.COLOR_BGR2RGB)
        fr_disp = front_disp_tf(image=fr_raw)["image"]
        tp_disp = top_disp_tf(image=tp_raw)["image"]
        fr_t = front_val_tf(image=fr_raw)["image"].unsqueeze(0).to(device)
        tp_t = top_val_tf(image=tp_raw)["image"].unsqueeze(0).to(device)

        cf, ct = j * 2, j * 2 + 1

        for row_idx, (bb, model) in enumerate(models_vis.items()):
            fr_attns = extract_attentions(model.backbone, fr_t)
            tp_attns = extract_attentions(model.backbone, tp_t)
            fr_heat = attn_to_heatmap(attention_rollout(fr_attns), grid_size, img_size)
            tp_heat = attn_to_heatmap(attention_rollout(tp_attns), grid_size, img_size)

            axes[row_idx, cf].imshow(fr_disp)
            axes[row_idx, cf].imshow(fr_heat, alpha=0.6, cmap="inferno")
            axes[row_idx, cf].set_title(f"{sid}\n{bb} Front", fontsize=7)
            axes[row_idx, cf].axis("off")

            axes[row_idx, ct].imshow(tp_disp)
            axes[row_idx, ct].imshow(tp_heat, alpha=0.6, cmap="inferno")
            axes[row_idx, ct].set_title(f"{bb} Top", fontsize=7)
            axes[row_idx, ct].axis("off")

    plt.tight_layout(); plt.show()

# 정리
for m in models_vis.values(): del m
models_vis.clear()
torch.cuda.empty_cache()
print("\n[INFO] DINOv2 → 건물 구조에 집중 (밝은 부분 = 구조물 영역)")
print("[INFO] EVA → 세부 불안정 요소에 집중 (배경 확산 → 과적합 위험)")
print("[INFO] 앙상블은 DINOv2의 구조물 포커스 + EVA의 분류 정밀도 결합")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 20. 오류 분석: 각 모델이 실패하는 패턴 분석
# ═══════════════════════════════════════════════════════════════════
import os, cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

os.chdir("/content/dacon")

# OOF 예측 사용 (stacking 셀에서 생성)
d_pred = (dino_oof_valid[:, 1] > 0.5).astype(int)
e_pred = (eva_oof_valid[:, 1] > 0.5).astype(int)

# --- Confusion Matrix ---
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, name, pred in [(axes[0], "DINOv2", d_pred), (axes[1], "EVA-Giant", e_pred)]:
    cm = confusion_matrix(valid_labels, pred, labels=[0, 1])
    disp = ConfusionMatrixDisplay(cm, display_labels=["Stable", "Unstable"])
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"{name} Confusion Matrix (OOF)")
plt.tight_layout(); plt.show()

# --- 에러 유형 분석 ---
d_correct = (d_pred == valid_labels)
e_correct = (e_pred == valid_labels)

error_groups = {
    "DINO만 틀림\n(EVA 정답)": ~d_correct & e_correct,
    "EVA만 틀림\n(DINO 정답)": d_correct & ~e_correct,
    "양쪽 모두 틀림": ~d_correct & ~e_correct,
}

# 에러 유형별 라벨 분포
print(f"{'='*70}")
print(f"  오류 패턴 상세 분석")
print(f"{'='*70}")
for name, mask in error_groups.items():
    n = mask.sum()
    if n == 0:
        print(f"\n  [{name.replace(chr(10), ' ')}]: 0개")
        continue
    labels_sub = valid_labels[mask]
    n_unstable = (labels_sub == 1).sum()
    n_stable = (labels_sub == 0).sum()
    print(f"\n  [{name.replace(chr(10), ' ')}]: {n}개")
    print(f"    실제 unstable: {n_unstable} ({n_unstable/n:.0%})")
    print(f"    실제 stable:   {n_stable} ({n_stable/n:.0%})")

    if name.startswith("DINO만"):
        # DINOv2가 틀린 경우 → 대부분 unstable을 stable로 오분류?
        dino_pred_sub = d_pred[mask]
        fn = ((dino_pred_sub == 0) & (labels_sub == 1)).sum()  # unstable → stable 오분류
        fp = ((dino_pred_sub == 1) & (labels_sub == 0)).sum()  # stable → unstable 오분류
        print(f"    DINOv2 False Negative (unstable→stable 오분류): {fn}")
        print(f"    DINOv2 False Positive (stable→unstable 오분류): {fp}")
    elif name.startswith("EVA만"):
        eva_pred_sub = e_pred[mask]
        fn = ((eva_pred_sub == 0) & (labels_sub == 1)).sum()
        fp = ((eva_pred_sub == 1) & (labels_sub == 0)).sum()
        print(f"    EVA False Negative (unstable→stable 오분류): {fn}")
        print(f"    EVA False Positive (stable→unstable 오분류): {fp}")

# --- 에러 샘플 이미지 시각화 ---
# all_df로부터 이미지 표시
train_df_local = pd.read_csv("data/open/train.csv")
n_train = len(train_df_local)

def get_image_path(global_idx):
    if global_idx < n_train:
        sid = train_df_local.iloc[global_idx]["id"]
        return os.path.join("data/open/train", sid)
    else:
        dev_idx = global_idx - n_train
        dev_df_local = pd.read_csv("data/open/dev.csv")
        sid = dev_df_local.iloc[dev_idx]["id"]
        return os.path.join("data/open/dev", sid)

# 유효 인덱스 매핑
valid_indices = np.where(both_valid)[0]

for err_name, err_mask in error_groups.items():
    n_err = err_mask.sum()
    if n_err == 0:
        continue
    n_show = min(4, n_err)
    err_local_idx = np.where(err_mask)[0][:n_show]

    fig, axes = plt.subplots(2, n_show, figsize=(4 * n_show, 6))
    if n_show == 1:
        axes = axes.reshape(2, 1)
    fig.suptitle(f"{err_name.replace(chr(10), ' ')} (상위 {n_show}개)", fontsize=12, fontweight="bold")

    for j, local_idx in enumerate(err_local_idx):
        global_idx = valid_indices[local_idx]
        img_dir = get_image_path(global_idx)
        true_label = "unstable" if valid_labels[local_idx] == 1 else "stable"
        d_prob = dino_oof_valid[local_idx, 1]
        e_prob = eva_oof_valid[local_idx, 1]

        front_path = os.path.join(img_dir, "front.png")
        top_path = os.path.join(img_dir, "top.png")
        if os.path.exists(front_path):
            fr = cv2.cvtColor(cv2.imread(front_path), cv2.COLOR_BGR2RGB)
            axes[0, j].imshow(fr)
        axes[0, j].set_title(f"Front\nTrue={true_label}\nDINO={d_prob:.3f} EVA={e_prob:.3f}", fontsize=8)
        axes[0, j].axis("off")

        if os.path.exists(top_path):
            tp = cv2.cvtColor(cv2.imread(top_path), cv2.COLOR_BGR2RGB)
            axes[1, j].imshow(tp)
        axes[1, j].set_title("Top", fontsize=8)
        axes[1, j].axis("off")

    plt.tight_layout(); plt.show()

# --- 핵심 인사이트 ---
print(f"\n{'='*60}")
print(f"  핵심 분석 결론")
print(f"{'='*60}")
print(f"  가설 검증:")
dino_fn = ((d_pred == 0) & (valid_labels == 1)).sum()
dino_fp = ((d_pred == 1) & (valid_labels == 0)).sum()
eva_fn = ((e_pred == 0) & (valid_labels == 1)).sum()
eva_fp = ((e_pred == 1) & (valid_labels == 0)).sum()
print(f"    DINOv2: FN={dino_fn} (unstable 놓침), FP={dino_fp} (stable 오분류)")
print(f"    EVA:    FN={eva_fn} (unstable 놓침), FP={eva_fp} (stable 오분류)")
if dino_fn > eva_fn:
    print(f"    → DINOv2가 unstable 분류에 약함 (가설 일치)")
if eva_fp > dino_fp:
    print(f"    → EVA가 stable을 unstable로 오분류 많음 (배경 의존 가설 일치)")
print(f"    → 앙상블로 양쪽 약점 보완 가능")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 21. Final Drive 동기화 + 정리
# ═══════════════════════════════════════════════════════════════════
import os, shutil, pickle

os.chdir("/content/dacon")
DRIVE_DIR = "/content/drive/MyDrive/dacon"
DRIVE_CKPT = os.path.join(DRIVE_DIR, "checkpoints")

print(f"{'='*60}")
print(f"  최종 동기화")
print(f"{'='*60}")

# 1) 양쪽 모델 체크포인트 → Drive
print("\n[1] DINOv2 체크포인트 → Drive")
sync_best_to_drive("dinov2_large")

print("\n[2] EVA-Giant 체크포인트 → Drive")
sync_best_to_drive("eva_giant")

# 2) 메타러너 → Drive
meta_local = os.path.join("checkpoints", "meta_learner.pkl")
meta_drive = os.path.join(DRIVE_CKPT, "meta_learner.pkl")
if os.path.exists(meta_local):
    shutil.copy2(meta_local, meta_drive)
    print(f"\n[3] 메타러너 → Drive: {meta_drive}")
else:
    print(f"\n[3] 메타러너 없음 (스태킹 셀 미실행)")

# 3) 로그 파일 → Drive
for bb in ["dinov2_large", "eva_giant"]:
    for f in os.listdir("checkpoints"):
        if f.startswith(bb) and f.endswith("_log.csv"):
            src = os.path.join("checkpoints", f)
            dst = os.path.join(DRIVE_CKPT, f)
            shutil.copy2(src, dst)
            print(f"  로그 → Drive: {f}")

# 4) 현황 요약
print(f"\n{'='*60}")
print(f"  최종 요약")
print(f"{'='*60}")

for bb in ["dinov2_large", "eva_giant"]:
    ckpts = sorted([f for f in os.listdir(DRIVE_CKPT)
                    if f.startswith(bb) and f.endswith('.pth') and '_ckpt' not in f])
    folds_done = [f"fold{i}" for i in range(5) if f"{bb}_fold{i}.pth" in ckpts]
    total_mb = sum(os.path.getsize(os.path.join(DRIVE_CKPT, c)) / 1e6 for c in ckpts)
    print(f"\n  {bb}:")
    print(f"    완료: {len(folds_done)}/5 folds {folds_done}")
    print(f"    크기: {total_mb:.0f} MB")

# 제출 파일
sub_dir = os.path.join(DRIVE_DIR, "submissions")
subs = sorted([f for f in os.listdir(sub_dir) if f.endswith('.csv')]) if os.path.isdir(sub_dir) else []
print(f"\n  제출 파일: {len(subs)}개")
for s in subs[-5:]:  # 최근 5개만
    print(f"    {s}")

free = shutil.disk_usage('/content').free / 1e9
print(f"\n  디스크 여유: {free:.1f} GB")
print(f"\n✅ 모든 작업 완료! Drive: {DRIVE_DIR}")
print(f"  → 체크포인트: {DRIVE_CKPT}")
print(f"  → 제출 파일: {sub_dir}")